In [ ]:
# Cell 1 — Setup
import sys
from pathlib import Path
import math
import torch
import numpy as np

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
)
from datasets import Dataset
import pandas as pd

# Setup Path per importare i moduli in src (stile notebook precedenti)
ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths

paths = get_paths(ROOT)
print("ROOT:", ROOT)
print("data_processed:", paths.data_processed)
print("checkpoints:", paths.checkpoints)


c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
TRAIN_PARQUET = paths.data_processed / "train_120k_ht.parquet"
VAL_PARQUET   = paths.data_processed / "val_120k_ht.parquet"

assert TRAIN_PARQUET.exists(), f"Missing: {TRAIN_PARQUET}"
assert VAL_PARQUET.exists(), f"Missing: {VAL_PARQUET}"

# Teacher checkpoint (come nel tuo notebook teacher)
TEACHER_DIR = paths.checkpoints / "teacher_bert_large_llrd/checkpoint-6000"
assert TEACHER_DIR.exists(), f"Missing teacher dir: {TEACHER_DIR}"

# Student pre-trained (google)
STUDENT_NAME = "google/bert_uncased_L-4_H-512_A-8"

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

def bf16_supported() -> bool:
    if not torch.cuda.is_available():
        return False
    # BF16 dipende dalla GPU; su molte RTX consumer è False.
    # Lasciamo una guardia semplice.
    try:
        return torch.cuda.is_bf16_supported()
    except Exception:
        return False

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3070


In [ ]:
# Cell 2 — Load parquet → HF Dataset

REQUIRED_COLS = ["input_ids", "attention_mask", "token_type_ids", "label"]

def load_flat_dataset(parquet_path: Path) -> Dataset:
    df = pd.read_parquet(parquet_path)
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in {parquet_path.name}: {missing}. Found: {list(df.columns)}")
    ds = Dataset.from_pandas(df[REQUIRED_COLS], preserve_index=False)
    # torch format: sequence columns restano tensori 1D per-sample; il collator farà padding dinamico.
    ds.set_format(type="torch", columns=REQUIRED_COLS)
    return ds

train_ds = load_flat_dataset(TRAIN_PARQUET)
val_ds   = load_flat_dataset(VAL_PARQUET)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))
print(train_ds[0].keys())


Train size: 96000
Val size: 12000
dict_keys(['input_ids', 'attention_mask', 'token_type_ids', 'label'])


In [ ]:
# Cell 3 — Tokenizer + Collator
# Nota: i tuoi parquet contengono già input_ids ecc. Il tokenizer serve solo per padding nel collator.

# Uso tokenizer bert-large-uncased per coerenza col teacher (vocab compatibile con student uncased)
tokenizer = AutoTokenizer.from_pretrained("bert-large-uncased", use_fast=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest", return_tensors="pt")


In [ ]:
# Cell 5 — Metriche (solo per monitoraggio)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def compute_metrics(p):
    # logits shape: [B, 1]
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int).reshape(-1)
    labels = p.label_ids.reshape(-1)

    acc = accuracy_score(labels, preds)
    pr, rc, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = 0.5

    return {"accuracy": acc, "f1": f1, "precision": pr, "recall": rc, "auc": auc}


In [ ]:
from td1_distillation import StudentWithProjections 

# Cell 6 — Modelli: teacher + student (wrapper con proiezioni)

# Teacher (fine-tuned)
teacher = AutoModelForSequenceClassification.from_pretrained(str(TEACHER_DIR))
teacher.eval()

# Student (pretrained piccolo). num_labels=1 (1 logit) coerente con BCEWithLogits in TD-2.
student_base = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=1)

# Opzionale: gradient checkpointing sullo student per risparmiare VRAM

#try:
 #   student_base.gradient_checkpointing_enable()
  #  print("Student gradient checkpointing: ENABLED")
#except Exception as e:
 #   print("Student gradient checkpointing not enabled:", e)
# Wrapper con proiezioni 512->1024
model = StudentWithProjections(student_base, student_hidden=512, teacher_hidden=1024)

# Move to device (Trainer lo farà, ma qui è comodo per sanity check)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
teacher.to(device)

print("Device:", device)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/bert_uncased_L-4_H-512_A-8 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Device: cuda


In [ ]:
# Cell 7 — LLRD (Layer-wise Learning Rate Decay) per lo student
from typing import List, Dict, Any
import re
import torch.nn as nn

def get_llrd_optimizer_parameters_bert(model: nn.Module, learning_rate: float, weight_decay: float, layer_decay: float = 0.95):
    """
    LLRD robusto per modelli BERT-like (anche piccoli).
    - classifier/pooler: LR massimo
    - encoder layers: LR decrescente dall'alto al basso
    - embeddings: LR minimo
    """
    named_parameters = list(model.named_parameters())
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]

    # Trova indici layer presenti (regex su encoder.layer.X.)
    layer_ids = set()
    pat = re.compile(r"encoder\.layer\.(\d+)\.")
    for n, _ in named_parameters:
        m = pat.search(n)
        if m:
            layer_ids.add(int(m.group(1)))
    if not layer_ids:
        raise ValueError("No encoder.layer.<idx> found in parameter names. Check model architecture.")
    max_layer = max(layer_ids)
    min_layer = min(layer_ids)
    num_layers = max_layer - min_layer + 1

    # Utility per evitare duplicati
    used = set()
    def pick(params):
        out = []
        for n, p in params:
            if n in used:
                continue
            used.add(n)
            out.append(p)
        return out

    opt_groups = []

    # 1) Classifier & pooler: LR max
    lr = learning_rate
    cls_params = [(n,p) for n,p in named_parameters if ("classifier" in n or "pooler" in n)]
    wd_params = [(n,p) for n,p in cls_params if not any(nd in n for nd in no_decay)]
    nd_params = [(n,p) for n,p in cls_params if any(nd in n for nd in no_decay)]
    if wd_params:
        opt_groups.append({"params": pick(wd_params), "weight_decay": weight_decay, "lr": lr})
    if nd_params:
        opt_groups.append({"params": pick(nd_params), "weight_decay": 0.0, "lr": lr})

    # 2) Encoder layers top->bottom (LR decrescente)
    # Iniziamo dal top layer (max_layer) con LR base, poi moltiplichiamo per layer_decay andando verso il basso.
    current_lr = learning_rate
    for layer in range(max_layer, min_layer - 1, -1):
        prefix = f"encoder.layer.{layer}."
        layer_params = [(n,p) for n,p in named_parameters if prefix in n]

        wd_params = [(n,p) for n,p in layer_params if not any(nd in n for nd in no_decay)]
        nd_params = [(n,p) for n,p in layer_params if any(nd in n for nd in no_decay)]

        if wd_params:
            opt_groups.append({"params": pick(wd_params), "weight_decay": weight_decay, "lr": current_lr})
        if nd_params:
            opt_groups.append({"params": pick(nd_params), "weight_decay": 0.0, "lr": current_lr})

        current_lr *= layer_decay

    # 3) Embeddings (LR minimo)
    emb_params = [(n,p) for n,p in named_parameters if "embeddings" in n]
    wd_params = [(n,p) for n,p in emb_params if not any(nd in n for nd in no_decay)]
    nd_params = [(n,p) for n,p in emb_params if any(nd in n for nd in no_decay)]
    if wd_params:
        opt_groups.append({"params": pick(wd_params), "weight_decay": weight_decay, "lr": current_lr})
    if nd_params:
        opt_groups.append({"params": pick(nd_params), "weight_decay": 0.0, "lr": current_lr})

    # Eventuali parametri non assegnati (safety net): LR minimo
    leftover = [(n,p) for n,p in named_parameters if n not in used]
    if leftover:
        wd_params = [(n,p) for n,p in leftover if not any(nd in n for nd in no_decay)]
        nd_params = [(n,p) for n,p in leftover if any(nd in n for nd in no_decay)]
        if wd_params:
            opt_groups.append({"params": pick(wd_params), "weight_decay": weight_decay, "lr": current_lr})
        if nd_params:
            opt_groups.append({"params": pick(nd_params), "weight_decay": 0.0, "lr": current_lr})

    return opt_groups

# Hyperparams TD-1
LR_MAX = 2e-5
DECAY_RATE = 0.95
WEIGHT_DECAY = 0.01

optimizer_grouped_parameters = get_llrd_optimizer_parameters_bert(
    model=model.student,  # LLRD sul backbone dello student (non sulle proiezioni)
    learning_rate=LR_MAX,
    weight_decay=WEIGHT_DECAY,
    layer_decay=DECAY_RATE
)

# Aggiungiamo anche le proiezioni (proj_emb, proj_hid) con LR_MAX (stesso lr del top)
proj_params = [p for n,p in model.named_parameters() if n.startswith("proj_")]
optimizer_grouped_parameters.append({"params": proj_params, "weight_decay": WEIGHT_DECAY, "lr": LR_MAX})

optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

print("Optimizer groups:", len(optimizer_grouped_parameters))


Optimizer groups: 13


In [ ]:
# Cell 8 — TrainingArguments (TD-1)

OUTPUT_DIR = paths.checkpoints / "student_bert4_td1_llrd"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Con BERT-large teacher + output_attentions, meglio partire conservativi su VRAM.
PER_DEVICE_BS = 1
GRAD_ACC_STEPS = 16 # effective batch ~ 16
EPOCHS = 8

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_BS,
    per_device_eval_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    num_train_epochs=EPOCHS,
    logging_steps=50,
    eval_strategy="epoch",
    #eval_steps=1000,
    save_strategy="epoch",
    #save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=not bf16_supported(),
    bf16=bf16_supported(),
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    prediction_loss_only=True
)
print(args)


ValueError: epochs is not a valid IntervalStrategy, please select one of ['no', 'steps', 'epoch']

In [ ]:
# Cell 9 — Trainer TD-1 + sanity check su 1 batch
from td1_distillation import TD1Trainer
trainer = TD1Trainer(
    model=model,
    args=args,
    data_collator=collator,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    teacher_model=teacher,
    # pesi loss TD-1 (modificabili)
    lambda_emb=1.0,
    lambda_hid=1.0,
    lambda_attn=0.5,
    optimizers=(optimizer, None),
)

# Sanity check: calcola loss su un batch piccolo
batch = collator([train_ds[i] for i in range(2)])  # anche se PER_DEVICE_BS=1, qui testiamo 2 esempi
batch = {k: v.to(model.device) for k,v in batch.items()}
loss_val = trainer.compute_loss(trainer.model, dict(batch), return_outputs=False)
print("Sanity loss (TD-1):", float(loss_val.detach().cpu()))


BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Sanity loss (TD-1): 5108.38671875


In [ ]:
# Cell 10 — Train + Evaluate + Save
train_result = trainer.train()
eval_result = trainer.evaluate()

print("Eval:", eval_result)

# Salva il best model (Trainer ha già best in memoria se load_best_model_at_end=True)
trainer.save_model(str(OUTPUT_DIR / "best"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "best"))

print("Saved best TD-1 student to:", OUTPUT_DIR / "best")


Step,Training Loss,Validation Loss
100,48858.575000,2492.201660
200,34715.332500,1982.675171


KeyboardInterrupt: 